# Engineering Decision-Intelligence Agent — Drive-Shaft Coupling `AMS-CPL-1042`

An agentic AI pipeline (built to the blueprint) that takes raw engineering documents in and returns an **auditable executive decision brief** out — not a summary.

**Architecture: Decompose → Tool-augment → Validate → Audit (with human oversight)**

| Stage | What runs |
|---|---|
| A. Extraction | One LLM extraction agent per document → typed Pydantic records (cheap model, structured outputs) |
| B. Normalisation | Deterministic unit + material-synonym normalisation |
| C. Comparison | Exhaustive field-by-field Rev A vs Rev B diff (schema-driven, not free text) |
| D. Validation | Intra-doc consistency, PO↔ECN effectivity, supplier compliance matrix |
| E. Quantitative | Deterministic tools: unit convert, tolerance range-check, capacity gap, finance |
| F. Synthesis | Human-in-the-loop gate on critical findings, then an evidence-constrained brief agent (strong model) |

**The five traps the pipeline must catch** (a naive "summarise these docs" prompt misses them):
1. Rev B is internally inconsistent — spec table says 42CrMo4, BOM table still says C45
2. PO-78214 orders obsolete Rev A with delivery *after* the ECN cutoff
3. The only compliant supplier (Beta) is 200 units/month short of demand
4. Gamma quotes in inches — `1.000 ±0.001 in` is actually *inside* `25.4 ±0.03 mm`
5. 1,420 Rev A units in stock must **not** be scrapped (ECN allows service use < 70 °C)

**To run:** you need an Anthropic API key (`https://platform.claude.com`). Add it in Colab via the 🔑 *Secrets* panel as `ANTHROPIC_API_KEY`, or you'll be prompted for it. Then just *Runtime → Run all* (one cell asks for a human decision — that's the human-in-the-loop gate, by design).

In [ ]:
# @title 1. Setup — install dependencies and configure the API key
%pip install -q anthropic pydantic

import os, json, datetime
from typing import Optional, List, Literal
from pydantic import BaseModel

# --- API key: Colab Secrets first, then env var, then prompt ---
if "ANTHROPIC_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # type: ignore
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

import anthropic
client = anthropic.Anthropic()

# Model routing (per blueprint §10): cheap model for extraction, strong model for synthesis
EXTRACT_MODEL = "claude-haiku-4-5"   # fast/cheap: reads documents into typed records
BRIEF_MODEL   = "claude-opus-4-8"    # strong: final judgement + executive brief

# --- Audit trail: every agent call, tool call, and human decision is logged ---
TRACE: list[dict] = []

def log(kind: str, name: str, detail):
    TRACE.append({
        "ts": datetime.datetime.now().isoformat(timespec="seconds"),
        "kind": kind,          # "agent" | "tool" | "validator" | "human" | "guard"
        "name": name,
        "detail": detail,
    })

print("Setup complete.")

In [ ]:
# @title 2. The case documents (raw inputs — the five traps are planted in here)
# In production these arrive as PDFs/scans via POST /cases/{id}/documents.
# Here they are embedded as text so the notebook runs end-to-end with zero uploads.

DOCUMENTS = {

"DRAWING_REV_A": """
ENGINEERING DRAWING — AMS-CPL-1042  Rev A          Sheet 1/1   Date: 2024-03-11
Title: Drive-Shaft Coupling

SPECIFICATION TABLE
  Bore diameter:        25.0 mm  tolerance +0.10 / -0.10 mm
  Material:             C45 (carbon steel)
  Surface finish (bore): Ra 3.2 um
  Hardening:            none
  Max. operating temp:  80 C
  Weight:               1.24 kg

COMPONENT / BOM TABLE
  Pos 1  Coupling body   Material: Carbon Steel C45   Qty 1
""",

"DRAWING_REV_B": """
ENGINEERING DRAWING — AMS-CPL-1042  Rev B          Sheet 1/1   Date: 2026-06-02
Title: Drive-Shaft Coupling
Change note: uprated for high-temperature duty (see ECN-2026-014)

SPECIFICATION TABLE
  Bore diameter:        25.4 mm  tolerance +0.03 / -0.03 mm
  Material:             42CrMo4 (alloy steel, quenched & tempered)
  Surface finish (bore): Ra 1.6 um
  Hardening:            induction hardened, 48-52 HRC
  Max. operating temp:  110 C
  Weight:               1.31 kg

COMPONENT / BOM TABLE
  Pos 1  Coupling body   Material: Carbon Steel C45   Qty 1
""",
# ^ TRAP 1: the BOM table was not updated — it still says C45.

"ECN_2026_014": """
ENGINEERING CHANGE NOTE  ECN-2026-014                     Issued: 2026-06-05
Part: AMS-CPL-1042  (Drive-Shaft Coupling)

1. Revision B is MANDATORY for all new production and all new purchase
   orders with delivery on or after 2026-08-01.
2. Revision A parts must NOT be used in new production after that date.
3. Existing Revision A stock MAY continue to be used for SERVICE /
   spare-part applications where operating temperature stays below 70 C.
4. Reason: field failures of Rev A couplings in high-temperature duty.
""",

"INVENTORY_REPORT": """
WAREHOUSE INVENTORY REPORT                                Date: 2026-07-15
Part number: AMS-CPL-1042
  Revision A on hand:  1,420 units   (bin W3-114, unrestricted stock)
  Revision B on hand:      0 units
Production planning note: forward demand for this part is 1,000 units/month.
""",

"SUPPLIER_QUOTES": """
RFQ-2291 QUOTE COMPARISON — AMS-CPL-1042 Rev B            Date: 2026-07-10

[1] Alpha Metallwerk GmbH        Unit price: EUR 14.20
    Claims revision: B
    Material offered: C45 carbon steel ("equivalent performance")
    Hardening: not offered
    Surface finish: Ra 3.2 um
    Capacity: 2,000 units/month
    Certificates: ISO 9001

[2] Beta Precision AG            Unit price: EUR 17.80
    Claims revision: B
    Material offered: 42CrMo4 Q+T
    Hardening: induction hardened 48-52 HRC
    Surface finish: Ra 1.6 um
    Bore: 25.4 mm +/-0.03 mm
    Capacity: 800 units/month
    Certificates: ISO 9001, IATF 16949, material certs 3.1

[3] Gamma Industries Inc. (USA)  Unit price: EUR 16.90
    Claims revision: B
    Material offered: 42CrMo4 (AISI 4140 equivalent)
    Hardening: induction hardening - process qualification to be confirmed
    Surface finish: to be confirmed
    Bore: 1.000 in +/-0.001 in
    Capacity: 600 units/month
    Certificates: material certs pending
""",
# ^ TRAP 3: only Beta is clearly compliant, but 800 < 1,000/month demand.
# ^ TRAP 4: Gamma quotes in inches — dimensionally compliant, but not obvious.

"PO_78214": """
PURCHASE ORDER  PO-78214                                  Date: 2026-07-08
Supplier: Alpha Metallwerk GmbH
Part: AMS-CPL-1042   Revision: A
Quantity: 1,000 units    Unit price: EUR 2.80
Requested delivery date: 2026-08-20
""",
# ^ TRAP 2: orders Rev A with delivery AFTER the 2026-08-01 mandatory-Rev-B date.

"QUALITY_REPORT": """
QUALITY FAILURE REPORT  QFR-2026-081                      Date: 2026-06-20
Part: AMS-CPL-1042  Revision A
Failure mode: bore wear + fatigue cracking in applications running above 80 C
Field failure rate: approx. 3 failures per month across the installed base
Average fully-loaded cost per field failure: EUR 8,333
  (replacement part, service call, downtime compensation)
Root cause: C45 material + untreated bore insufficient for high-temp duty.
Recommendation: material and hardening upgrade (implemented as Revision B).
""",
}

for doc_id, text in DOCUMENTS.items():
    print(f"{doc_id:18s} {len(text):5d} chars")

In [ ]:
# @title 3. Canonical data model — typed records with provenance (blueprint §2)
# Downstream agents reason over these records, never over raw documents.
# spec-table material and BOM-table material are SEPARATE fields on purpose:
# that separation is what makes trap 1 detectable instead of averaged away.

class Measure(BaseModel):
    value: float
    unit: str                       # as stated in the document ("mm", "in")
    tol_plus: Optional[float] = None
    tol_minus: Optional[float] = None

class PartSpec(BaseModel):
    part_number: str
    revision: str
    bore: Measure
    material_spec_table: str        # material as stated in the SPECIFICATION table
    material_bom_table: str         # material as stated in the COMPONENT/BOM table
    surface_finish_ra_um: float
    hardening: Optional[str] = None # None if the drawing says no hardening
    max_temp_c: float
    weight_kg: float

class ECNRecord(BaseModel):
    ecn_id: str
    part_number: str
    mandatory_revision: str
    effective_date: str             # ISO date on/after which the revision is mandatory
    old_revision_allowed_in_new_production: bool
    service_use_allowed_below_c: Optional[float]

class InventoryRecord(BaseModel):
    part_number: str
    revision: str
    quantity_on_hand: int
    monthly_demand_units: int

class SupplierOffer(BaseModel):
    supplier: str
    price_eur: float
    revision_claimed: str
    material: Optional[str] = None
    hardening: Optional[str] = None       # None if not offered / not stated
    surface_finish_ra_um: Optional[float] = None  # None if "to be confirmed"
    bore: Optional[Measure] = None        # in the units the supplier quoted
    capacity_per_month: Optional[int] = None
    certificates: Optional[str] = None

class SupplierQuotes(BaseModel):
    offers: List[SupplierOffer]

class PORecord(BaseModel):
    po_number: str
    supplier: str
    part_number: str
    revision_ordered: str
    quantity: int
    unit_price_eur: float
    delivery_date: str              # ISO

class QualityRecord(BaseModel):
    part_number: str
    revision: str
    failure_mode: str
    failures_per_month: float
    avg_cost_per_failure_eur: float

class Finding(BaseModel):
    severity: Literal["critical", "warning", "info"]
    category: str                   # consistency | compliance | supply | finance | inventory | quality
    message: str
    evidence: List[str]             # doc ids / quoted values backing the claim
    recommended_action: str
    requires_human: bool = False

FINDINGS: List[Finding] = []

def add_finding(f: Finding):
    FINDINGS.append(f)
    log("validator", f.category, f.model_dump())
    flag = "  [NEEDS HUMAN]" if f.requires_human else ""
    print(f"[{f.severity.upper():8s}] {f.category}: {f.message}{flag}")

print("Schemas defined.")

In [ ]:
# @title 4. Deterministic tool layer — where correctness actually lives (blueprint §4)
# The LLM decides WHAT to check; these pure functions compute THE ANSWER.
# The LLM never does arithmetic, unit math, or date comparisons.

MM_PER_UNIT = {"mm": 1.0, "in": 25.4, "inch": 25.4, '"': 25.4}

def convert_to_mm(m: Measure) -> Measure:
    """Normalise a length Measure to millimetres (SI on ingest, blueprint rule)."""
    f = MM_PER_UNIT[m.unit.lower().strip()]
    out = Measure(
        value=round(m.value * f, 4), unit="mm",
        tol_plus=round(m.tol_plus * f, 4) if m.tol_plus is not None else None,
        tol_minus=round(m.tol_minus * f, 4) if m.tol_minus is not None else None,
    )
    log("tool", "convert_to_mm", {"in": m.model_dump(), "out": out.model_dump()})
    return out

def check_tolerance(actual: Measure, required: Measure) -> str:
    """PASS if the actual tolerance band lies entirely inside the required band.
    A RANGE check, not an equality check — 25.400 +/-0.0254 fits inside 25.4 +/-0.03."""
    a, r = convert_to_mm(actual), convert_to_mm(required)
    a_lo, a_hi = a.value - a.tol_minus, a.value + a.tol_plus
    r_lo, r_hi = r.value - r.tol_minus, r.value + r.tol_plus
    verdict = "PASS" if (a_lo >= r_lo and a_hi <= r_hi) else "FAIL"
    log("tool", "check_tolerance",
        {"actual_mm": [a_lo, a_hi], "required_mm": [r_lo, r_hi], "verdict": verdict})
    return verdict

def check_effectivity(po: PORecord, ecn: ECNRecord) -> Optional[dict]:
    """Cross-document date/revision logic. Returns a conflict dict, or None if OK."""
    d_delivery = datetime.date.fromisoformat(po.delivery_date)
    d_effective = datetime.date.fromisoformat(ecn.effective_date)
    conflict = (po.revision_ordered != ecn.mandatory_revision) and (d_delivery >= d_effective)
    result = None
    if conflict:
        result = {
            "po": po.po_number, "revision_ordered": po.revision_ordered,
            "delivery_date": po.delivery_date,
            "mandatory_revision": ecn.mandatory_revision,
            "effective_date": ecn.effective_date,
        }
    log("tool", "check_effectivity", {"po": po.po_number, "conflict": result})
    return result

def capacity_gap(demand_per_month: int, compliant_capacity_per_month: int) -> int:
    gap = max(0, demand_per_month - compliant_capacity_per_month)
    log("tool", "capacity_gap",
        {"demand": demand_per_month, "capacity": compliant_capacity_per_month, "gap": gap})
    return gap

def financial_delta(old_price: float, new_price: float, monthly_volume: int,
                    failures_avoided_per_month: float, cost_per_failure: float) -> dict:
    """Annual incremental part cost vs annual avoided failure cost. An ESTIMATE
    on stated assumptions — the brief must label it as such."""
    incremental_cost = round((new_price - old_price) * monthly_volume * 12)
    avoided_cost = round(failures_avoided_per_month * cost_per_failure * 12)
    result = {
        "annual_incremental_part_cost_eur": incremental_cost,
        "annual_avoided_failure_cost_eur": avoided_cost,
        "annual_net_benefit_eur": avoided_cost - incremental_cost,
        "assumptions": (f"volume {monthly_volume}/month, price {old_price} -> {new_price} EUR, "
                        f"{failures_avoided_per_month} failures/month avoided at {cost_per_failure} EUR each"),
    }
    log("tool", "financial_delta", result)
    return result

def diff_specs(rev_a: PartSpec, rev_b: PartSpec) -> List[dict]:
    """Exhaustive field-by-field comparison over the schema — it CANNOT
    'compare one row and stop' because it iterates every model field."""
    deltas = []
    for field in PartSpec.model_fields:
        if field in ("part_number", "revision"):
            continue
        va, vb = getattr(rev_a, field), getattr(rev_b, field)
        if isinstance(va, Measure):
            va, vb = convert_to_mm(va).model_dump(), convert_to_mm(vb).model_dump()
        else:
            va = va.model_dump() if isinstance(va, BaseModel) else va
            vb = vb.model_dump() if isinstance(vb, BaseModel) else vb
        deltas.append({"attribute": field, "rev_a": va, "rev_b": vb, "changed": va != vb})
    log("tool", "diff_specs", {"changed": [d["attribute"] for d in deltas if d["changed"]]})
    return deltas

# --- unit tests: tools are trusted only because they are tested ---
assert convert_to_mm(Measure(value=1.0, unit="in", tol_plus=0.001, tol_minus=0.001)).value == 25.4
assert check_tolerance(Measure(value=1.0, unit="in", tol_plus=0.001, tol_minus=0.001),
                       Measure(value=25.4, unit="mm", tol_plus=0.03, tol_minus=0.03)) == "PASS"
assert check_tolerance(Measure(value=25.0, unit="mm", tol_plus=0.10, tol_minus=0.10),
                       Measure(value=25.4, unit="mm", tol_plus=0.03, tol_minus=0.03)) == "FAIL"
assert capacity_gap(1000, 800) == 200
print("Deterministic tools defined and self-tested.")

In [ ]:
# @title 5. Stage A — extraction agents: one LLM call per document → typed record
# Structured outputs guarantee schema-valid records (a malformed extraction fails
# fast instead of poisoning the brief). The extractor copies values VERBATIM in
# the units stated — normalisation is the tool layer's job, never the LLM's.

EXTRACTION_RULES = """You are a document-extraction agent for engineering documents.
Rules (non-negotiable):
- Copy every value VERBATIM from the document. Do not convert units, do not
  round, do not infer values that are not stated.
- If a field is stated as 'none' / 'not offered' / 'to be confirmed' / absent,
  return null for that field rather than guessing.
- The specification table and the component/BOM table are extracted into
  SEPARATE fields. Do not reconcile them if they disagree — report both as-is.
- Dates must be returned in ISO format (YYYY-MM-DD)."""

def extract(doc_id: str, schema: type[BaseModel]) -> BaseModel:
    response = client.messages.parse(
        model=EXTRACT_MODEL,
        max_tokens=2048,
        system=EXTRACTION_RULES,
        messages=[{
            "role": "user",
            "content": f"Extract the structured record from this document "
                       f"(doc_id={doc_id}):\n\n{DOCUMENTS[doc_id]}",
        }],
        output_format=schema,
    )
    record = response.parsed_output
    log("agent", f"extract:{doc_id}",
        {"model": EXTRACT_MODEL, "schema": schema.__name__, "output": record.model_dump()})
    return record

spec_a    = extract("DRAWING_REV_A", PartSpec)
spec_b    = extract("DRAWING_REV_B", PartSpec)
ecn       = extract("ECN_2026_014", ECNRecord)
inventory = extract("INVENTORY_REPORT", InventoryRecord)
quotes    = extract("SUPPLIER_QUOTES", SupplierQuotes).offers
po        = extract("PO_78214", PORecord)
quality   = extract("QUALITY_REPORT", QualityRecord)

print("\n--- Extracted records ---")
for name, rec in [("Rev A", spec_a), ("Rev B", spec_b), ("ECN", ecn),
                  ("Inventory", inventory), ("PO", po), ("Quality", quality)]:
    print(f"\n{name}: {rec.model_dump_json(indent=2)}")
print("\nSupplier offers:")
for o in quotes:
    print(f"  {o.model_dump_json()}")

In [ ]:
# @title 6. Stages B+C — normalisation and exhaustive Rev A vs Rev B comparison

def normalize_material(name: Optional[str]) -> Optional[str]:
    """Synonym resolution: 'Carbon Steel C45' == 'C45', '42CrMo4 Q+T' == '42CrMo4'."""
    if name is None:
        return None
    n = name.lower()
    if "42crmo4" in n.replace(" ", "") or "4140" in n:
        return "42CrMo4"
    if "c45" in n:
        return "C45"
    return name.strip()

revision_deltas = diff_specs(spec_a, spec_b)

print(f"Revision comparison  {spec_a.part_number}  Rev {spec_a.revision} -> Rev {spec_b.revision}\n")
print(f"{'attribute':24s} {'Rev A':32s} {'Rev B':32s} changed")
for d in revision_deltas:
    print(f"{d['attribute']:24s} {str(d['rev_a']):32.32s} {str(d['rev_b']):32.32s} "
          f"{'<-- CHANGED' if d['changed'] else ''}")

In [ ]:
# @title 7. Stage D — validators (this is where the traps get caught)
# Validators FLAG conflicts, they never silently pick a value (blueprint §7:
# "never average C45 and 42CrMo4 into 'steel'").

# --- Validator 1: intra-document consistency (TRAP 1) -----------------------
if normalize_material(spec_b.material_spec_table) != normalize_material(spec_b.material_bom_table):
    add_finding(Finding(
        severity="critical", category="consistency",
        message=(f"Rev {spec_b.revision} drawing is internally inconsistent: spec table says "
                 f"'{spec_b.material_spec_table}' but component/BOM table says "
                 f"'{spec_b.material_bom_table}'. Release must be frozen until engineering "
                 f"confirms the correct material."),
        evidence=["DRAWING_REV_B: spec table material", "DRAWING_REV_B: BOM table material"],
        recommended_action="Freeze Rev B release; obtain engineering clarification of the true material.",
        requires_human=True,
    ))

# --- Validator 2: cross-document effectivity, PO vs ECN (TRAP 2) ------------
conflict = check_effectivity(po, ecn)
if conflict:
    add_finding(Finding(
        severity="critical", category="compliance",
        message=(f"{po.po_number} orders obsolete Rev {po.revision_ordered} from {po.supplier} "
                 f"with delivery {po.delivery_date}, which is AFTER the mandatory Rev "
                 f"{ecn.mandatory_revision} effectivity date {ecn.effective_date} ({ecn.ecn_id})."),
        evidence=[f"PO_78214: revision {po.revision_ordered}, delivery {po.delivery_date}",
                  f"ECN_2026_014: Rev {ecn.mandatory_revision} mandatory from {ecn.effective_date}"],
        recommended_action="Put PO-78214 on hold and amend it to Revision B (or re-source).",
        requires_human=True,
    ))

# --- Validator 3: supplier compliance matrix, attribute by attribute --------
required_material = normalize_material(spec_b.material_spec_table)

def assess_offer(o: SupplierOffer) -> dict:
    checks = {}
    checks["material"] = ("OK" if normalize_material(o.material) == required_material
                          else "FAIL" if o.material else "TO_VERIFY")
    if o.hardening is None:
        checks["hardening"] = "FAIL" if spec_b.hardening else "OK"
    else:
        checks["hardening"] = "TO_VERIFY" if "confirm" in o.hardening.lower() else "OK"
    if o.surface_finish_ra_um is None:
        checks["surface_finish"] = "TO_VERIFY"
    else:
        checks["surface_finish"] = "OK" if o.surface_finish_ra_um <= spec_b.surface_finish_ra_um else "FAIL"
    # Dimensional conformance runs through the TOOL, never through prose reasoning.
    # This is what makes Gamma's inch quote comparable at all (TRAP 4).
    checks["bore_tolerance"] = ("TO_VERIFY" if o.bore is None
                                else check_tolerance(o.bore, spec_b.bore))
    checks["bore_tolerance"] = {"PASS": "OK", "FAIL": "FAIL"}.get(checks["bore_tolerance"], checks["bore_tolerance"])
    checks["certificates"] = ("TO_VERIFY" if not o.certificates or "pending" in o.certificates.lower()
                              else "OK")
    if any(v == "FAIL" for v in checks.values()):
        status = "NON-COMPLIANT"
    elif any(v == "TO_VERIFY" for v in checks.values()):
        status = "TO-VERIFY"
    else:
        status = "COMPLIANT"
    return {"supplier": o.supplier, "price_eur": o.price_eur,
            "capacity_per_month": o.capacity_per_month, "checks": checks, "status": status}

supplier_matrix = [assess_offer(o) for o in quotes]
log("validator", "supplier_compliance_matrix", supplier_matrix)

print("\nSupplier compliance matrix (vs Rev B requirements):")
for row in supplier_matrix:
    print(f"\n  {row['supplier']}  ({row['status']}, EUR {row['price_eur']}, "
          f"{row['capacity_per_month']}/month)")
    for attr, verdict in row["checks"].items():
        print(f"    {attr:16s} {verdict}")

for row in supplier_matrix:
    if row["status"] == "NON-COMPLIANT":
        fails = [a for a, v in row["checks"].items() if v == "FAIL"]
        add_finding(Finding(
            severity="warning", category="compliance",
            message=f"{row['supplier']} claims Rev B but is non-compliant on: {', '.join(fails)}.",
            evidence=[f"SUPPLIER_QUOTES: {row['supplier']}"],
            recommended_action=f"Reject {row['supplier']}'s offer for Rev B in its current form.",
        ))
    elif row["status"] == "TO-VERIFY":
        opens = [a for a, v in row["checks"].items() if v == "TO_VERIFY"]
        add_finding(Finding(
            severity="warning", category="compliance",
            message=(f"{row['supplier']} is dimensionally compliant (tolerance range-check PASS "
                     f"after unit conversion) but not yet fully qualified: open items {', '.join(opens)}."),
            evidence=[f"SUPPLIER_QUOTES: {row['supplier']}", "tool: convert_to_mm + check_tolerance"],
            recommended_action=f"Accelerate qualification of {row['supplier']} (hardening process, finish, certificates).",
        ))

In [ ]:
# @title 8. Stage E — quantitative reasoning (tool-backed, never model arithmetic)

# --- Capacity (TRAP 3): demand vs capacity of clearly-COMPLIANT suppliers only ---
compliant_capacity = sum(r["capacity_per_month"] or 0
                         for r in supplier_matrix if r["status"] == "COMPLIANT")
gap = capacity_gap(inventory.monthly_demand_units, compliant_capacity)
if gap > 0:
    compliant_names = [r["supplier"] for r in supplier_matrix if r["status"] == "COMPLIANT"]
    add_finding(Finding(
        severity="critical", category="supply",
        message=(f"Compliant supply ({', '.join(compliant_names)}: {compliant_capacity}/month) "
                 f"covers only part of demand ({inventory.monthly_demand_units}/month) — "
                 f"shortfall of {gap} units/month. Awarding the business does not solve supply."),
        evidence=["SUPPLIER_QUOTES: capacities", "INVENTORY_REPORT: demand 1,000/month",
                  "tool: capacity_gap"],
        recommended_action=(f"Confirm compliant capacity in writing and accelerate qualification "
                            f"of a second source for the {gap}/month gap."),
    ))

# --- Finance: incremental cost vs avoided failure cost (indicative estimate) ---
beta_price = next(r["price_eur"] for r in supplier_matrix if r["status"] == "COMPLIANT")
economics = financial_delta(
    old_price=po.unit_price_eur,                    # current Rev A contract price
    new_price=beta_price,                           # compliant Rev B offer
    monthly_volume=inventory.monthly_demand_units,
    failures_avoided_per_month=quality.failures_per_month,
    cost_per_failure=quality.avg_cost_per_failure_eur,
)
add_finding(Finding(
    severity="info", category="finance",
    message=(f"Indicative economics: ~EUR {economics['annual_incremental_part_cost_eur']:,}/yr "
             f"incremental part cost vs ~EUR {economics['annual_avoided_failure_cost_eur']:,}/yr "
             f"avoided failure cost = ~EUR {economics['annual_net_benefit_eur']:,}/yr net benefit. "
             f"ESTIMATE on stated assumptions: {economics['assumptions']}."),
    evidence=["PO_78214: current unit price", "SUPPLIER_QUOTES: Beta price",
              "QUALITY_REPORT: failure rate and cost", "tool: financial_delta"],
    recommended_action="Treat as indicative; refine with negotiated pricing before final approval.",
))

# --- Inventory disposition (TRAP 5): rule-driven, with a forbidden-conclusion guard ---
disposition = []
if not ecn.old_revision_allowed_in_new_production:
    disposition.append("block from new production")
disposition.append("segregate in warehouse (move out of unrestricted stock)")
if ecn.service_use_allowed_below_c is not None:
    disposition.append(f"retain for approved service use below "
                       f"{ecn.service_use_allowed_below_c:g} C per {ecn.ecn_id}")
disposition_text = "; ".join(disposition)

# Guard: the exact LLM failure the blueprint calls out — 'scrap all' is forbidden.
assert "scrap" not in disposition_text.lower(), "FORBIDDEN CONCLUSION: scrapping is not permitted"
log("guard", "no_scrap_all", {"disposition": disposition_text, "passed": True})

add_finding(Finding(
    severity="warning", category="inventory",
    message=(f"{inventory.quantity_on_hand:,} Rev {inventory.revision} units on hand: "
             f"{disposition_text}. Do NOT scrap — the ECN explicitly permits service use."),
    evidence=["INVENTORY_REPORT: 1,420 Rev A units, unrestricted stock",
              "ECN_2026_014: service use allowed below 70 C"],
    recommended_action=disposition_text,
))

print(f"\nCapacity gap: {gap}/month   Net benefit: EUR {economics['annual_net_benefit_eur']:,}/yr (estimate)")

In [ ]:
# @title 9. Human-in-the-loop gate — critical findings block the brief (blueprint §8)
# The system will NOT emit a "release Rev B" recommendation while an unresolved
# critical exception exists. It surfaces the exception and waits for a human.

exceptions = [f for f in FINDINGS if f.severity == "critical" and f.requires_human]
resolutions = []

if exceptions:
    print(f"{len(exceptions)} critical finding(s) require human resolution before the brief "
          f"can be issued.\n")
    reviewer = input("Your name (for the audit trail): ").strip() or "unnamed reviewer"
    for i, f in enumerate(exceptions, 1):
        print(f"\n--- Exception {i}/{len(exceptions)} [{f.category}] ---")
        print(f.message)
        print(f"Evidence: {f.evidence}")
        print(f"Proposed action: {f.recommended_action}")
        decision = input("Your resolution/decision (e.g. 'confirmed 42CrMo4 is correct; "
                         "update BOM' or 'hold PO-78214, amend to Rev B'): ").strip()
        resolution = {
            "finding_category": f.category,
            "finding_message": f.message,
            "decision": decision or f"accepted proposed action: {f.recommended_action}",
            "decided_by": reviewer,
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        }
        resolutions.append(resolution)
        log("human", "exception_resolution", resolution)
    print("\nAll critical exceptions resolved and logged. Gate open — brief can be generated.")
else:
    print("No critical findings requiring human resolution. Gate open.")

In [ ]:
# @title 10. Stage F — executive brief agent (strong model, evidence-constrained)
# The anti-hallucination boundary: the brief agent may ONLY use the findings and
# tool results already in state. It cannot introduce a claim without evidence.

brief_state = {
    "part": {"part_number": spec_b.part_number, "from_revision": spec_a.revision,
             "to_revision": spec_b.revision},
    "revision_deltas": [d for d in revision_deltas if d["changed"]],
    "findings": [f.model_dump() for f in FINDINGS],
    "human_resolutions": resolutions,
    "supplier_matrix": supplier_matrix,
    "quantitative": {
        "monthly_demand_units": inventory.monthly_demand_units,
        "compliant_capacity_per_month": compliant_capacity,
        "capacity_gap_per_month": gap,
        "economics": economics,
        "inventory_on_hand_rev_a": inventory.quantity_on_hand,
        "inventory_disposition": disposition_text,
    },
}

BRIEF_SYSTEM = """You are the Executive Brief Agent in an engineering
decision-intelligence pipeline. You write the final management brief.

Hard constraints:
- Use ONLY the facts in the JSON state provided. Do not add any fact, number,
  supplier, date, or claim that is not present in the state.
- Every number you state must appear verbatim in the state.
- Cite the evidence document ids in square brackets, e.g. [DRAWING_REV_B],
  after each factual claim.
- Label the economics explicitly as an estimate on stated assumptions.
- Reflect the human resolutions: critical exceptions were reviewed by a named
  human, and the recommendation must be conditional on those resolutions being
  executed (e.g. 'after engineering clarification', 'after PO amendment').

Structure the brief exactly as:
SITUATION - two or three sentences on what changed from Rev A to Rev B and why.
CRITICAL FINDINGS - numbered list, most severe first.
SUPPLIER ASSESSMENT - one line per supplier with status.
INDICATIVE ECONOMICS - the estimate, with its assumptions.
RECOMMENDED ACTIONS - split into 'Immediate' and 'Within 1 week'.
MANAGEMENT DECISION - one short paragraph with the recommended decision."""

response = client.messages.create(
    model=BRIEF_MODEL,
    max_tokens=8000,
    thinking={"type": "adaptive"},
    system=BRIEF_SYSTEM,
    messages=[{"role": "user",
               "content": "Case state (sole source of truth):\n\n"
                          + json.dumps(brief_state, indent=2)}],
)
brief_text = next(b.text for b in response.content if b.type == "text")
log("agent", "executive_brief", {"model": BRIEF_MODEL, "brief": brief_text})

# Grounding spot-check: the brief must engage with all five traps.
must_mention = ["C45", "42CrMo4", "PO-78214", "200", "Gamma", "1,420"]
missing = [m for m in must_mention if m not in brief_text.replace("1420", "1,420")]
if missing:
    print(f"!! Groundedness check FAILED — brief does not mention: {missing}\n")
else:
    print("Groundedness check passed: all five traps are addressed in the brief.\n")

print("=" * 78)
print(f"EXECUTIVE DECISION BRIEF — {spec_b.part_number}  Rev {spec_a.revision} -> {spec_b.revision}")
print("=" * 78)
print(brief_text)

In [ ]:
# @title 11. Audit trail — every agent I/O, tool call, and human decision
# This is what turns the output from "an AI said so" into "an auditable decision".

print(f"{len(TRACE)} trace entries\n")
print(f"{'#':>3} {'time':19s} {'kind':10s} name")
for i, t in enumerate(TRACE):
    print(f"{i:3d} {t['ts']:19s} {t['kind']:10s} {t['name']}")

# Full detail is available for any entry, e.g.:
print("\nExample — full detail of the tolerance check on Gamma's inch quote:")
gamma_check = [t for t in TRACE if t["kind"] == "tool" and t["name"] == "check_tolerance"][-1]
print(json.dumps(gamma_check, indent=2))

# Persist the complete audit trail alongside the brief.
with open("audit_trail.json", "w") as fh:
    json.dump({"case": spec_b.part_number, "trace": TRACE,
               "findings": [f.model_dump() for f in FINDINGS],
               "human_resolutions": resolutions,
               "brief": brief_text}, fh, indent=2)
print("\nSaved full audit trail + brief to audit_trail.json")